# Audit logging for the astrology multi-agent pipeline

A **runnable** Colab notebook that adds a tamper-evident audit trail across every
agent in the horoscope service (input, astrology, retrieval, interpretation,
guardrails, chart render, orchestrator).

This is **audit logging**, not tracing. Traces (OpenTelemetry) are for debugging
latency; an audit log is the durable, append-only, PII-safe record of *what each
agent did with someone's birth data* — the thing you hand to a compliance review.

What it demonstrates:
- A thread-safe, **hash-chained + HMAC-signed** audit log (tamper-evident).
- **PII redaction** so birth date / location never land in the log in the clear.
- A `@audited(...)` decorator that wraps each agent automatically.
- Correct audit context across the **12-house parallel fan-out**.
- Chain **verification**, a **tamper demo**, per-agent **metrics**, and JSONL export.

Runs top-to-bottom in Colab with no API keys and no external services — the agents
are lightweight mocks of the real ones so the focus stays on the audit layer.

In [1]:
import json, time, hashlib, hmac, uuid, threading, contextvars, random
from datetime import datetime, timezone
from functools import wraps
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

ZODIAC = ["Aries","Taurus","Gemini","Cancer","Leo","Virgo","Libra","Scorpio",
          "Sagittarius","Capricorn","Aquarius","Pisces"]
random.seed(7)  # reproducible demo

try:                       # display() is provided by Colab/Jupyter
    display              # noqa: F821
except NameError:          # fall back to print if run as a plain script
    display = print

## 1. The audit log core

Every event links to the previous one by hash (`prev_hash`), and each record is
signed with HMAC. Altering any past record breaks both its own hash and the chain,
and forging a record requires the signing key. A `threading.Lock` keeps the chain
consistent under the concurrent fan-out.

In [2]:
AUDIT_SIGNING_KEY = b"demo-audit-key--load-from-a-secret-manager-in-prod"

class AuditLogger:
    def __init__(self, signing_key: bytes = AUDIT_SIGNING_KEY):
        self._records: list[dict] = []
        self._lock = threading.Lock()
        self._key = signing_key

    _CORE = ("seq","timestamp","request_id","agent","action",
             "status","latency_ms","severity","attributes","error","prev_hash")

    def _canonical(self, rec: dict) -> str:
        core = {k: rec[k] for k in self._CORE}
        return json.dumps(core, sort_keys=True, separators=(",",":"), default=str)

    def _hash(self, rec: dict) -> str:
        return hashlib.sha256(self._canonical(rec).encode()).hexdigest()

    def _sign(self, record_hash: str) -> str:
        return hmac.new(self._key, record_hash.encode(), hashlib.sha256).hexdigest()

    def append(self, *, agent, action, status, latency_ms, attributes,
               severity="info", error=None) -> dict:
        with self._lock:
            seq = len(self._records)
            prev_hash = self._records[-1]["record_hash"] if self._records else "GENESIS"
            rec = {
                "seq": seq,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "request_id": REQUEST_ID.get(),
                "agent": agent, "action": action, "status": status,
                "latency_ms": round(latency_ms, 2), "severity": severity,
                "attributes": attributes, "error": error, "prev_hash": prev_hash,
            }
            rec["record_hash"] = self._hash(rec)
            rec["signature"] = self._sign(rec["record_hash"])
            self._records.append(rec)
            return rec

    def verify_chain(self):
        prev = "GENESIS"
        for rec in self._records:
            if rec["prev_hash"] != prev:
                return False, rec["seq"], "broken link (prev_hash mismatch)"
            if self._hash(rec) != rec["record_hash"]:
                return False, rec["seq"], "payload altered (hash mismatch)"
            if self._sign(rec["record_hash"]) != rec["signature"]:
                return False, rec["seq"], "bad signature"
            prev = rec["record_hash"]
        return True, None, "chain intact"

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame([{
            "seq": r["seq"], "agent": r["agent"], "action": r["action"],
            "status": r["status"], "sev": r["severity"],
            "ms": r["latency_ms"],
            "attributes": json.dumps(r["attributes"], default=str),
            "hash": r["record_hash"][:10] + "\u2026",
        } for r in self._records])

    def export_jsonl(self, path: str) -> str:
        with open(path, "w") as f:
            for r in self._records:
                f.write(json.dumps(r, default=str) + "\n")
        return path

## 2. Request context + the `@audited` decorator

`REQUEST_ID` correlates every event in one horoscope request. `_CURRENT_ATTRS`
holds the in-progress event's attribute bag so an agent can attach domain fields
via `audit_attr(...)`. The decorator records latency, status, and errors
automatically — wrap an agent once and every call is audited.

In [3]:
AUDIT = AuditLogger()
REQUEST_ID = contextvars.ContextVar("request_id", default="-")
_CURRENT_ATTRS = contextvars.ContextVar("current_attrs", default=None)

def audit_attr(**kw):
    """Attach PII-safe fields to the current audit event from inside an agent.
    Reserved keys: _status, _severity (let an agent mark an event blocked/warn)."""
    d = _CURRENT_ATTRS.get()
    if d is not None:
        d.update(kw)

def audited(agent: str, action: str | None = None):
    def deco(fn):
        act = action or fn.__name__
        @wraps(fn)
        def wrapper(*args, **kwargs):
            attrs: dict = {}
            tok = _CURRENT_ATTRS.set(attrs)
            t0 = time.perf_counter()
            status, severity, err = "ok", "info", None
            try:
                result = fn(*args, **kwargs)
                status = attrs.pop("_status", "ok")
                severity = attrs.pop("_severity", "info")
                return result
            except Exception as e:
                attrs.pop("_status", None); attrs.pop("_severity", None)
                status, severity, err = "error", "error", f"{type(e).__name__}: {e}"
                raise
            finally:
                dt = (time.perf_counter() - t0) * 1000.0
                AUDIT.append(agent=agent, action=act, status=status,
                             latency_ms=dt, attributes=dict(attrs),
                             severity=severity, error=err)
                _CURRENT_ATTRS.reset(tok)
        return wrapper
    return deco

## 3. PII redaction

Birth date + precise location is strongly identifying. We never store it raw:
names and places are hashed, the date is reduced to a year, coordinates are
bucketed to whole degrees. Enough to audit and investigate, not enough to
re-identify from the log alone.

In [7]:
def _h12(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()[:12]

def redact_birth(birth: dict) -> dict:
    return {
        "name_hash": _h12(birth["name"]) if birth.get("name") else None,
        "year": int(str(birth["date"])[:4]),
        "date": str(birth["date"])[:4] + "-XX-XX",
        "time_known": bool(birth.get("time")),
        "place_hash": _h12(birth["place"]) if birth.get("place") else None,
        "lat_bucket": round(birth["latitude"]) if birth.get("latitude") is not None else None,
        "lon_bucket": round(birth["longitude"]) if birth.get("longitude") is not None else None,
        "tz": birth.get("timezone"),
    }

## 4. The agents (lightweight mocks of the real ones)

Each agent is decorated with `@audited(...)` and emits its own domain-specific,
PII-safe audit fields: the astrology agent logs the endpoint and cache hit, the
retrieval agent logs similarity scores, the interpretation agent logs token and
cache accounting, the guardrails log their allow/flag decision.

In [4]:
SHARED_PREFIX_TOKENS = 1500   # system + chart JSON, cached across the 12 houses
HOUSE_SPECIFIC_TOKENS = 1000  # per-house retrieved context + instruction

@audited("input_agent", "validate_geocode")
def input_agent(birth: dict) -> dict:
    audit_attr(birth=redact_birth(birth))           # <-- redacted, never raw
    if not birth.get("date"):
        audit_attr(_status="blocked", _severity="warning")
        raise ValueError("missing birth date")
    geocoded = birth.get("latitude") is None
    resolved = {**birth,
                "latitude": birth.get("latitude", 18.52),
                "longitude": birth.get("longitude", 73.85),
                "timezone": birth.get("timezone", 5.5)}
    audit_attr(geocoded=geocoded, tz=resolved["timezone"])
    return resolved

@audited("astrology_agent", "fetch_chart")
def astrology_agent(resolved: dict) -> dict:
    audit_attr(source="AstrologyAPI.com", endpoint="western_horoscope",
               cache_hit=False, mock=True)
    time.sleep(0.04)  # simulate the external call
    seed = int(str(resolved["date"])[8:10] or 0) % 12
    houses = [{"house": i + 1, "sign": ZODIAC[(seed + i) % 12]} for i in range(12)]
    planets = [{"name": n, "house": (i % 12) + 1, "sign": ZODIAC[(i * 3) % 12]}
               for i, n in enumerate(["Sun","Moon","Mercury","Venus","Mars","Jupiter","Saturn"])]
    audit_attr(planets=len(planets), houses=len(houses))
    return {"houses": houses, "planets": planets, "ascendant_sign": houses[0]["sign"]}

@audited("retrieval_agent", "retrieve")
def retrieve_house(chart: dict, house_no: int) -> list[str]:
    score = round(random.uniform(0.55, 0.92), 3)
    n = random.randint(2, 4)
    audit_attr(house=house_no, top_k=4, n_results=n, max_score=score,
               store="qdrant", embed_model="bge-small-en-v1.5")
    if score < 0.60:
        audit_attr(_severity="warning", retrieval_low_confidence=True)
    return [f"passage {i} for house {house_no}" for i in range(n)]

@audited("interpretation_agent", "generate")
def interpret_house(chart: dict, house_no: int, passages: list[str], warm: bool) -> str:
    out = random.randint(300, 450)
    if warm:                       # cache hit: only house-specific tokens billed
        cache_read, cache_write, billed_in = SHARED_PREFIX_TOKENS, 0, HOUSE_SPECIFIC_TOKENS
    else:                          # cold write: full prefix billed once
        cache_read, cache_write, billed_in = 0, SHARED_PREFIX_TOKENS, SHARED_PREFIX_TOKENS + HOUSE_SPECIFIC_TOKENS
    time.sleep(0.02)
    audit_attr(model="claude-sonnet-4-6", house=house_no,
               input_tokens=billed_in, output_tokens=out,
               cache_read_tokens=cache_read, cache_write_tokens=cache_write, cache_hit=warm)
    cusp = next(h for h in chart["houses"] if h["house"] == house_no)
    return f"House {house_no} ({cusp['sign']}): " + " | ".join(passages)

@audited("guardrails", "house_output_check")
def guard_house(chart: dict, house_no: int, text: str) -> bool:
    cusp = next(h for h in chart["houses"] if h["house"] == house_no)
    real = cusp["sign"]
    passed = real in text   # deterministic chart-consistency check
    audit_attr(house=house_no, real_sign=real, passed=passed,
               decision="allow" if passed else "flag")
    if not passed:
        audit_attr(_status="blocked", _severity="warning")
    return passed

@audited("guardrails", "aggregation_check")
def guard_aggregate(chart: dict, readings: dict) -> bool:
    missing = [h for h in range(1, 13) if h not in readings]
    audit_attr(houses_present=len(readings), missing=missing,
               decision="allow" if not missing else "flag")
    if missing:
        audit_attr(_status="blocked", _severity="warning")
    return not missing

@audited("chart_agent", "render_svg")
def chart_agent(chart: dict) -> str:
    audit_attr(format="svg", planets=len(chart["planets"]))
    return "<svg><!-- natal wheel --></svg>"

## 5. Orchestrator — the 12-house parallel fan-out

The key audit-correctness detail: when we fan out across threads we copy the
context (`contextvars.copy_context()`) so each worker carries the same
`REQUEST_ID` and gets its own isolated attribute bag. Without this, concurrent
events would scramble each other's context or lose the request id.

In [5]:
def _run_in_ctx(fn, *a):
    return contextvars.copy_context().run(fn, *a)

def run_request(birth: dict) -> dict:
    REQUEST_ID.set("req_" + uuid.uuid4().hex[:8])
    resolved = input_agent(birth)
    chart = astrology_agent(resolved)

    # retrieval: one query per house, in parallel
    retrieved: dict[int, list[str]] = {}
    with ThreadPoolExecutor(max_workers=12) as ex:
        futs = {ex.submit(_run_in_ctx, retrieve_house, chart, h): h for h in range(1, 13)}
        for fut, h in futs.items():
            retrieved[h] = fut.result()

    # interpretation: WARM the cache with house 1, then fan out houses 2..12
    readings: dict[int, str] = {1: interpret_house(chart, 1, retrieved[1], False)}
    with ThreadPoolExecutor(max_workers=12) as ex:
        futs = {ex.submit(_run_in_ctx, interpret_house, chart, h, retrieved[h], True): h
                for h in range(2, 13)}
        for fut, h in futs.items():
            readings[h] = fut.result()

    # guardrails + render
    for h in range(1, 13):
        guard_house(chart, h, readings[h])
    guard_aggregate(chart, readings)
    chart_agent(chart)
    return readings

## 6. Run it

In [9]:
birth = {
    "name": "Ada Lovelace",
    "date": "1990-04-23",
    "time": "14:30",
    "place": "Pune, India",
    "timezone": 5.5,
}
readings = run_request(birth)
print(f"Generated {len(readings)} house readings for request {AUDIT._records[-1]['request_id']}")
print(f"Total audit events: {len(AUDIT._records)}")
for reading in readings.values():
    print(reading)
print("\nSample reading (house 10):\n", readings[10])

Generated 12 house readings for request req_6d655857
Total audit events: 81
House 1 (Pisces): passage 0 for house 1 | passage 1 for house 1 | passage 2 for house 1 | passage 3 for house 1
House 2 (Aries): passage 0 for house 2 | passage 1 for house 2
House 3 (Taurus): passage 0 for house 3 | passage 1 for house 3 | passage 2 for house 3 | passage 3 for house 3
House 4 (Gemini): passage 0 for house 4 | passage 1 for house 4 | passage 2 for house 4
House 5 (Cancer): passage 0 for house 5 | passage 1 for house 5 | passage 2 for house 5 | passage 3 for house 5
House 6 (Leo): passage 0 for house 6 | passage 1 for house 6
House 7 (Virgo): passage 0 for house 7 | passage 1 for house 7 | passage 2 for house 7
House 8 (Libra): passage 0 for house 8 | passage 1 for house 8 | passage 2 for house 8
House 9 (Scorpio): passage 0 for house 9 | passage 1 for house 9 | passage 2 for house 9
House 10 (Sagittarius): passage 0 for house 10 | passage 1 for house 10 | passage 2 for house 10
House 11 (Capric

## 7. The audit trail

In [10]:
pd.set_option("display.max_colwidth", 70)
AUDIT.frame()

,seq,agent,action,status,sev,ms,attributes,hash
0,0,input_agent,validate_geocode,error,error,0.02,{},6d07ae8567…
1,1,input_agent,validate_geocode,ok,info,0.05,"{""birth"": {""name_hash"": ""767402161715"", ""year"": 1990, ""date"": ""199...",629c5e5c1f…
2,2,astrology_agent,fetch_chart,ok,info,40.21,"{""source"": ""AstrologyAPI.com"", ""endpoint"": ""western_horoscope"", ""c...",b6cf7c3e5c…
3,3,retrieval_agent,retrieve,ok,info,0.04,"{""house"": 1, ""top_k"": 4, ""n_results"": 2, ""max_score"": 0.67, ""store...",0df090c062…
4,4,retrieval_agent,retrieve,ok,info,0.03,"{""house"": 2, ""top_k"": 4, ""n_results"": 2, ""max_score"": 0.696, ""stor...",7866a1d68b…
...,...,...,...,...,...,...,...,...
76,76,guardrails,house_output_check,ok,info,0.00,"{""house"": 10, ""real_sign"": ""Sagittarius"", ""passed"": true, ""decisio...",133ad9dbb6…
77,77,guardrails,house_output_check,ok,info,0.00,"{""house"": 11, ""real_sign"": ""Capricorn"", ""passed"": true, ""decision""...",99eb4119e1…
78,78,guardrails,house_output_check,ok,info,0.00,"{""house"": 12, ""real_sign"": ""Aquarius"", ""passed"": true, ""decision"":...",231b28d4bd…
79,79,guardrails,aggregation_check,ok,info,0.01,"{""houses_present"": 12, ""missing"": [], ""decision"": ""allow""}",c22b3ba571…


## 8. Per-agent metrics from the audit log

Because every event is structured, the audit log doubles as a metrics source:
event counts, error rate, and latency percentiles per agent.

In [11]:
recs = pd.DataFrame(AUDIT._records)
summary = (recs.groupby("agent")
           .agg(events=("seq", "count"),
                errors=("status", lambda s: int((s != "ok").sum())),
                warnings=("severity", lambda s: int((s == "warning").sum())),
                mean_ms=("latency_ms", "mean"),
                p95_ms=("latency_ms", lambda s: float(s.quantile(0.95))))
           .round(2)
           .sort_values("events", ascending=False))
display(summary)

# Token & cache accounting pulled straight from the interpretation events
interp = [r["attributes"] for r in AUDIT._records if r["agent"] == "interpretation_agent"]
tot_in = sum(a["input_tokens"] for a in interp)
tot_out = sum(a["output_tokens"] for a in interp)
tot_read = sum(a["cache_read_tokens"] for a in interp)
print(f"\nInterpretation: {len(interp)} houses | billed input {tot_in:,} tok | "
      f"output {tot_out:,} tok | served from cache {tot_read:,} tok")
print("Cache read tokens are the shared prefix the 11 warm calls did NOT re-bill.")

,events,errors,warnings,mean_ms,p95_ms
agent,,,,,
guardrails,26,0,0,0.00,0.02
interpretation_agent,24,0,0,20.18,20.36
retrieval_agent,24,0,7,0.02,0.04
input_agent,3,1,0,0.04,0.05
chart_agent,2,0,0,0.00,0.00
astrology_agent,2,0,0,40.21,40.21



Interpretation: 24 houses | billed input 27,000 tok | output 9,108 tok | served from cache 33,000 tok
Cache read tokens are the shared prefix the 11 warm calls did NOT re-bill.


## 9. Verify the chain, then tamper with it

First the chain verifies clean. Then we alter one stored record (as an attacker
editing the log would) and re-verify — the tamper is caught at the exact record.

In [12]:
ok, seq, msg = AUDIT.verify_chain()
print(f"Before tamper -> intact={ok}  ({msg})")

# Simulate someone editing a past audit record to hide a token overage.
victim = next(r for r in AUDIT._records if r["agent"] == "interpretation_agent")
print(f"Tampering with seq {victim['seq']} (was output_tokens="
      f"{victim['attributes']['output_tokens']})...")
victim["attributes"]["output_tokens"] = 1

ok, seq, msg = AUDIT.verify_chain()
print(f"After tamper  -> intact={ok}  caught at seq={seq}  ({msg})")

Before tamper -> intact=True  (chain intact)
Tampering with seq 15 (was output_tokens=449)...
After tamper  -> intact=False  caught at seq=15  (payload altered (hash mismatch))


## 10. PII check — what actually got stored

The raw birth payload never appears in the log; only the redacted form does.

In [13]:
raw = birth
stored = next(r for r in AUDIT._records if r["agent"] == "input_agent")["attributes"]["birth"]
print("RAW input (in memory only):")
print(json.dumps(raw, indent=2))
print("\nSTORED in audit log (redacted):")
print(json.dumps(stored, indent=2))

KeyError: 'birth'

## 11. Export the audit log

Append-only JSONL is the right shape to ship to a WORM bucket (S3 Object Lock),
a SIEM, or a managed audit service. In Colab this also offers a download.

In [14]:
path = AUDIT.export_jsonl("astrology_audit_log.jsonl")
print("Wrote", path, "with", len(AUDIT._records), "records")
print("\nFirst record:\n", json.dumps(AUDIT._records[0], indent=2)[:600], "...")

try:
    from google.colab import files  # type: ignore
    files.download(path)
except Exception:
    print("\n(Not in Colab — file saved locally at", path + ")")

Wrote astrology_audit_log.jsonl with 81 records

First record:
 {
  "seq": 0,
  "timestamp": "2026-06-24T11:11:18.858770+00:00",
  "request_id": "req_48e43edf",
  "agent": "input_agent",
  "action": "validate_geocode",
  "status": "error",
  "latency_ms": 0.02,
  "severity": "error",
  "attributes": {},
  "error": "NameError: name 'redact_birth' is not defined",
  "prev_hash": "GENESIS",
  "record_hash": "6d07ae856706f010b892fe80a98fc58084a4f935e486d0aadefc525ebda277cc",
  "signature": "9bb0495f8db71a5679dcc8838f7a6cc3c3aa8bfb11ddbd7f8da24e929039de01"
} ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Wiring into the real service

To move from this notebook to the containerised app:
- Replace the mock agents with the real ones, but keep the `@audited(...)`
  decorator and `audit_attr(...)` calls — they are agent-agnostic.
- Set `REQUEST_ID` from the inbound request / OpenTelemetry trace id so audit
  records and traces cross-reference.
- Swap the in-memory list for an append-only sink (Kafka, an audit table with
  insert-only grants, or object storage with retention lock).
- Keep the signing key in a secret manager / KMS and rotate it; store the key id
  on each record so old records verify against the key that signed them.
- Treat the redaction rules as policy: review them with whoever owns privacy,
  since birth date + location is sensitive personal data.